Import libraries

In [1]:
from lxml import html, etree
import pandas as pd
import pyperclip, math

Declare namespaces

In [2]:
ns = {'svg': 'http://www.w3.org/2000/svg'}

Calculate parallel lines

In [3]:
def parallel_lines(x1, y1, x2, y2, items):
    """
    items = [
        {"class": "line", "width": 5},
        {"class": "connected", "width": 3},
        ...
    ]

    Computes parallel SVG lines with class-based widths and no gaps.
    """

    dx = x2 - x1
    dy = y2 - y1
    length = math.hypot(dx, dy)

    # Perpendicular unit vector
    px = -dy / length
    py =  dx / length

    if not items:
        return []

    n = len(items)
    offsets = [0] * n

    # Extract widths
    widths = [item["width"] for item in items]

    # --- Compute offsets based on actual widths ---
    if n == 1:
        offsets[0] = 0

    elif n % 2 == 1:
        # Odd number → center line exists
        mid = n // 2
        offsets[mid] = 0

        # Left side
        cum = 0
        for i in range(mid - 1, -1, -1):
            cum += widths[i + 1]
            offsets[i] = -cum

        # Right side
        cum = 0
        for i in range(mid + 1, n):
            cum += widths[i - 1]
            offsets[i] = +cum

    else:
        # Even number → no center line
        half = n // 2

        # Left side
        cum = 0
        for i in range(half - 1, -1, -1):
            cum += widths[i]
            offsets[i] = -cum + widths[i] / 2

        # Right side
        cum = 0
        for i in range(half, n):
            cum += widths[i]
            offsets[i] = cum - widths[i] / 2

    # --- Build SVG lines ---
    svg_lines = []
    for off, item in zip(offsets, items):
        myClass = item["class"]
        w   = item["width"]

        x1p = x1 + px * off
        y1p = y1 + py * off
        x2p = x2 + px * off
        y2p = y2 + py * off

        svg_lines.append(
            f'<line x1="{x1p}" y1="{y1p}" x2="{x2p}" y2="{y2p}" class="{myClass}" stroke-width="{w}"><title>{myClass}</title></line>'
        )

    return svg_lines

Process connections table with navgraph svg in connections html

In [ ]:
myHTML = html.parse('manuscripts.html')
nodeGroup = myHTML.xpath('//g[@id="nodes"]')[0]
#print(nodeGroup)
navgraphNodeIds = [node.get('id') for node in nodeGroup.findall('g', ns)]
#print(f'Node IDs: {navgraphNodeIds}')
connectionGroup = etree.fromstring(f'<g class="connections" id="manuscripts" display="block"/>')

connectionDf = pd.read_csv('https://docs.google.com/spreadsheets/d/15mjhEAJDR5rhdux3mdjVVLWATm20uo_droIn5K4kRXA/export?format=csv&gid=1532912149')
dataCols = connectionDf.columns[connectionDf.notna().any()][2:8]
for index, row in connectionDf.iterrows():
    myFromId = row['from']
    if myFromId not in navgraphNodeIds:
        print(f'!!!{myFromId} not in Navgraph!!!')
        continue
    myToId = row['to']
    if myToId not in navgraphNodeIds:
        print(f'!!!{myToId} not in Navgraph!!!')
        continue
    
    # print(f'From {myFromId} to {myToId}:')
    myNodeFrom = nodeGroup.find(f'g[@id="{myFromId}"]', ns)
    cx_from = myNodeFrom.find('circle', ns).get('cx')
    cy_from = myNodeFrom.find('circle', ns).get('cy')

    myNodeTo = nodeGroup.find(f'g[@id="{myToId}"]', ns)
    cx_to = myNodeTo.find('circle', ns).get('cx')
    cy_to = myNodeTo.find('circle', ns).get('cy')
    
    myItems = []
        
    for col in dataCols:
        dataCell = row[col]
        if pd.notna(dataCell):
            myItems.append({
                'class': col,
                'width': 2
            });
            
    myLine = parallel_lines(float(cx_from), float(cy_from), float(cx_to), float(cy_to), myItems)
    for myLine in myLine:
        # print(f'Line: {myLine}')
        connectionGroup.append(etree.fromstring(myLine))
outputGroup = etree.tostring(connectionGroup, encoding='unicode', pretty_print=True)
# print(outputGroup)
pyperclip.copy(outputGroup) #copy to clipboard